# Self-guided lesson — Working with dates in pandas

Dates often arrive in a dataset as **strings**. Before we can sort them chronologically, calculate durations, filter by time periods, or extract information such as the month or weekday, we usually need to convert them to a proper datetime type.

In this lesson you will learn how to:

- inspect a column containing dates;
- convert strings with `pd.to_datetime()`;
- handle invalid dates safely;
- work with ambiguous date formats;
- extract date components with `.dt`;
- calculate differences and add time intervals;
- read dates directly from a CSV;
- filter data using dates.

The goal is not to memorize every datetime method. Focus on the workflow:

**inspect → convert → validate → derive → use**

In [ ]:
import pandas as pd

## 1. Dates are often strings

Consider a small dataset containing timestamps written in European day-month-year format.

In [ ]:
df = pd.DataFrame({
    "event": ["A", "B", "C"],
    "date": [
        "01-09-2018 15:23:11",
        "02-09-2018 17:21:45",
        "15-10-2018 09:05:00"
    ]
})

df

In [ ]:
df.dtypes

The values *look* like dates to us, but pandas currently sees the `date` column as text (`object`).

That distinction matters. A string does not automatically understand chronological order or date arithmetic.

## 2. Converting strings with `pd.to_datetime()`

When the format is known, specifying it explicitly is usually the safest option.

For our strings:

- `%d` → day
- `%m` → month
- `%Y` → four-digit year
- `%H` → hour (24-hour clock)
- `%M` → minute
- `%S` → second

In [ ]:
df["date"] = pd.to_datetime(
    df["date"],
    format="%d-%m-%Y %H:%M:%S"
)

df

In [ ]:
df.dtypes

The column now has a pandas datetime type (`datetime64[ns]`).

### Why specify the format?

Dates such as `03-04-2025` are ambiguous: they could mean 3 April or March 4 depending on the convention being used.

If you know the format of your source data, state it explicitly rather than asking pandas to guess.

## 3. Handling invalid dates

Real datasets may contain malformed or impossible dates. `errors="coerce"` is useful during cleaning because invalid values become `NaT` (**Not a Time**) instead of stopping the whole conversion.

In [ ]:
dirty_dates = pd.DataFrame({
    "raw_date": [
        "03/10/2000",
        "not-a-date",
        "31/02/2000",
        "18/12/2000"
    ]
})

dirty_dates["parsed_date"] = pd.to_datetime(
    dirty_dates["raw_date"],
    format="%d/%m/%Y",
    errors="coerce"
)

dirty_dates

In [ ]:
dirty_dates["parsed_date"].isna()

`NaT` should not automatically be deleted. First investigate **why** the value could not be parsed and decide what makes sense for the project.

This follows the same cleaning principle used elsewhere in the course:

**detect the problem → understand it → decide → validate**

## 4. When you know the convention but not an exact format

Sometimes strings are consistently written with the day first but may not all have exactly the same representation. In those situations, `dayfirst=True` can communicate the intended convention.

In [ ]:
dates = pd.Series(["05/01/2025", "18/02/2025", "27/03/2025"])

pd.to_datetime(dates, dayfirst=True)

Prefer an explicit `format=` when the source has a known, stable format. Use options such as `dayfirst=True` when they accurately describe the source data.

## 5. Extracting information with `.dt`

Once a Series is datetime-like, pandas provides the `.dt` accessor.

In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["hour"] = df["date"].dt.hour
df["weekday"] = df["date"].dt.day_name()

df

Derived date features can be useful in analysis. For example, you may want to compare activity by month, weekday, or hour.

Before creating a feature, ask whether it helps answer your analytical question.

## 6. Formatting a datetime for display

`.dt.strftime()` converts a datetime into a formatted **string**.

In [ ]:
df["date_label"] = df["date"].dt.strftime("%d/%m/%Y")

df[["date", "date_label"]]

In [ ]:
df[["date", "date_label"]].dtypes

Notice the important difference:

- `date` remains a datetime and supports date operations;
- `date_label` is text intended for display.

Do not replace a useful datetime column with a formatted string unless you actually need to.

## 7. Date arithmetic

Datetime values understand calendar arithmetic. We can add a duration or calculate the difference between two dates.

In [ ]:
df["three_days_later"] = df["date"] + pd.Timedelta(days=3)

df[["date", "three_days_later"]]

In [ ]:
sessions = pd.DataFrame({
    "start": pd.to_datetime([
        "2025-01-10 09:15:00",
        "2025-01-10 11:30:00"
    ]),
    "end": pd.to_datetime([
        "2025-01-10 10:05:00",
        "2025-01-10 13:00:00"
    ])
})

sessions["duration"] = sessions["end"] - sessions["start"]
sessions

In [ ]:
sessions["duration_minutes"] = sessions["duration"].dt.total_seconds() / 60

sessions

This is much safer than manually subtracting hours and minutes.

> `Timedelta` is ideal for fixed durations such as days, hours, minutes, or seconds. Calendar operations such as “one month later” need special care because months do not all have the same length.

## 8. Reading dates from a CSV

`read_csv()` can parse selected columns as dates while loading the data.

The following public dataset contains international football results.

In [ ]:
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/results.csv"

football = pd.read_csv(url)
football[["date", "home_team", "away_team"]].head()

In [ ]:
football["date"] = pd.to_datetime(
    football["date"],
    format="%Y-%m-%d",
    errors="coerce"
)

football.dtypes

Another possibility is `parse_dates=` when reading the CSV:

```python
football = pd.read_csv(url, parse_dates=["date"])
```

For a data-cleaning project, converting explicitly after loading can be easier to inspect and validate because you see the raw values first.

## 9. Filtering by date

Once the column has been converted, date conditions become straightforward.

In [ ]:
start_date = pd.Timestamp("2022-01-01")
end_date = pd.Timestamp("2022-12-31")

mask = football["date"].between(start_date, end_date)

football_2022 = football.loc[mask]
football_2022.head()

In [ ]:
football_2022["date"].min(), football_2022["date"].max()

The final line is an example of **validation**: after filtering, check that the resulting range is actually the one you intended.

# 💡 Check for understanding

Use the football shootouts dataset below.

In [ ]:
shootouts_url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/shootouts.csv"

shootouts = pd.read_csv(shootouts_url)
shootouts.head()

Complete the following tasks:

1. Inspect the data types. Is `date` already a datetime?
2. Convert `date` to a datetime using the appropriate format.
3. Validate the conversion by checking the data type and the minimum and maximum dates.
4. Create columns containing:
   - `year`
   - `month`
   - `weekday`
5. Find how many shootouts occurred in each year.
6. Filter the dataset to keep only shootouts from 2020 onwards.
7. Validate your filter by checking the earliest date in the result.

**Optional:** Which weekday contains the largest number of recorded shootouts?

# Summary

A reliable workflow for dates is:

1. **Inspect** the original values and their data type.
2. **Understand the source format** instead of guessing.
3. **Convert** with `pd.to_datetime()`.
4. Use `errors="coerce"` when malformed values need to be identified safely.
5. **Validate** the conversion and investigate `NaT` values.
6. Use `.dt` to derive useful date components.
7. Use datetime arithmetic for durations and date differences.
8. Filter and analyze dates only after confirming that the conversion is correct.

The key idea is simple: a date stored as text may look correct, but converting it to a datetime gives pandas the temporal logic needed to work with it reliably.